# Transfer-Performance Matching
This notebook focuses on creating performance block/age/season/transfer value pairs,
which can ultimately be used to train and test a supervised Bayesian linear regression
model to predict transfer values.

In [1]:
import pandas as pd
import duckdb
from understatapi import UnderstatClient
import numpy as np
import os
import sys

# Also import local helpers
target_dir = os.path.abspath('..') 

if target_dir not in sys.path:
    sys.path.append(target_dir)
    
from preprocess import merge_stats_df_with_transfermarkt

In [2]:
# 1. Get position data
positions = ['F']
f_stats_df = pd.read_csv(f"../data/{'_'.join(positions)}_stats_2.csv")


# Train/test split by player
all_players = f_stats_df['player_id'].unique()

# 80/20 split
np.random.seed(42)
train_players = np.random.choice(all_players, size=int(0.8*len(all_players)), replace=False)

train_df = f_stats_df[f_stats_df['player_id'].isin(train_players)]
test_df = f_stats_df[~f_stats_df['player_id'].isin(train_players)]

train_df.head()

,player_id,player_name,date,goals_per_90,xG_per_90,assists_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90,league
0,140,Yannick Gerhardt,2015-11-21 18:30:00,0.207852,0.076191,0.000000,0.152804,1.454965,0.276498,0.165737,Bundesliga
1,140,Yannick Gerhardt,2016-03-05 18:30:00,0.111940,0.147187,0.111940,0.097708,1.119403,0.419527,0.204866,Bundesliga
2,140,Yannick Gerhardt,2021-08-14 13:30:00,0.000000,0.004439,0.412214,0.149183,0.961832,0.423672,0.274489,Bundesliga
3,140,Yannick Gerhardt,2021-12-17 19:30:00,0.000000,0.038612,0.000000,0.122557,0.432000,0.267215,0.123039,Bundesliga
4,140,Yannick Gerhardt,2022-04-03 13:30:00,0.000000,0.143876,0.000000,0.016857,0.623557,0.317156,0.181113,Bundesliga


In [3]:
train_df.head()

,player_id,player_name,date,goals_per_90,xG_per_90,assists_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90,league
0,140,Yannick Gerhardt,2015-11-21 18:30:00,0.207852,0.076191,0.000000,0.152804,1.454965,0.276498,0.165737,Bundesliga
1,140,Yannick Gerhardt,2016-03-05 18:30:00,0.111940,0.147187,0.111940,0.097708,1.119403,0.419527,0.204866,Bundesliga
2,140,Yannick Gerhardt,2021-08-14 13:30:00,0.000000,0.004439,0.412214,0.149183,0.961832,0.423672,0.274489,Bundesliga
3,140,Yannick Gerhardt,2021-12-17 19:30:00,0.000000,0.038612,0.000000,0.122557,0.432000,0.267215,0.123039,Bundesliga
4,140,Yannick Gerhardt,2022-04-03 13:30:00,0.000000,0.143876,0.000000,0.016857,0.623557,0.317156,0.181113,Bundesliga


In [4]:
# 2. Pull transfer value data and merge
train_df_combined = merge_stats_df_with_transfermarkt(train_df, use_transfermarkt_info=True)
test_df_combined = merge_stats_df_with_transfermarkt(test_df, use_transfermarkt_info=True)

/Users/lancehendricks/Documents/College Coding/ML 2/TransferValues/src/preprocess/player_stats.py:217: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stats_df["date"] = stats_df['date'].astype('datetime64[us]')
/Users/lancehendricks/Documents/College Coding/ML 2/TransferValues/src/preprocess/player_stats.py:217: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stats_df["date"] = stats_df['date'].astype('datetime64[us]')


In [5]:
train_df_combined.head()

,player_id,player_name,date,date_of_birth,league,goals_per_90,xG_per_90,assists_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90,value
7588,1513.0,Mauro Icardi,2014-11-08,1993-02-19,Serie_A,0.590551,0.667127,0.118110,0.154321,1.062992,0.693518,0.182508,21000000
7608,1294.0,Paulo Dybala,2014-11-08,1993-11-15,Serie_A,0.320665,0.327707,0.213777,0.119004,1.603325,0.514270,0.223899,6500000
8240,2284.0,Daniel Wass,2014-11-20,1989-05-31,Ligue_1,0.411429,0.213534,0.205714,0.123267,1.440000,0.249786,0.106858,5000000
8267,3277.0,Alexandre Lacazette,2014-11-20,1991-05-28,Ligue_1,0.800000,0.541697,0.300000,0.180375,1.700000,0.722309,0.254882,17000000
8336,3696.0,Morgan Sanson,2014-11-20,1994-08-18,Ligue_1,0.105510,0.120888,0.105510,0.099815,1.055100,0.311706,0.112268,3500000


In [6]:
# Add age and year (fractional columns
train_df_combined["age"] = (train_df_combined["date"] - train_df_combined["date_of_birth"]).dt.days  / 365.25 # .25 accounts for leap years
test_df_combined["age"] = (test_df_combined["date"] - test_df_combined["date_of_birth"]).dt.days  / 365.25


train_df_combined["year"] = train_df_combined["date"].dt.year + (train_df_combined["date"].dt.day_of_year / 365.25)
test_df_combined["year"] = test_df_combined["date"].dt.year + (test_df_combined["date"].dt.day_of_year / 365.25)

train_df_combined.head()

,player_id,player_name,date,date_of_birth,league,goals_per_90,xG_per_90,assists_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90,value,age,year
7588,1513.0,Mauro Icardi,2014-11-08,1993-02-19,Serie_A,0.590551,0.667127,0.118110,0.154321,1.062992,0.693518,0.182508,21000000,21.716632,2014.854209
7608,1294.0,Paulo Dybala,2014-11-08,1993-11-15,Serie_A,0.320665,0.327707,0.213777,0.119004,1.603325,0.514270,0.223899,6500000,20.980151,2014.854209
8240,2284.0,Daniel Wass,2014-11-20,1989-05-31,Ligue_1,0.411429,0.213534,0.205714,0.123267,1.440000,0.249786,0.106858,5000000,25.472964,2014.887064
8267,3277.0,Alexandre Lacazette,2014-11-20,1991-05-28,Ligue_1,0.800000,0.541697,0.300000,0.180375,1.700000,0.722309,0.254882,17000000,23.482546,2014.887064
8336,3696.0,Morgan Sanson,2014-11-20,1994-08-18,Ligue_1,0.105510,0.120888,0.105510,0.099815,1.055100,0.311706,0.112268,3500000,20.257358,2014.887064


In [7]:
# Get rid of DOB column
train_df_combined = train_df_combined.drop("date_of_birth", axis=1)
test_df_combined = test_df_combined.drop("date_of_birth", axis=1)

In [8]:
# Save to csv
train_df_combined.to_csv(f"../data/{'_'.join(positions)}_stats_values_train.csv", index=False)
test_df_combined.to_csv(f"../data/{'_'.join(positions)}_stats_values_test.csv", index=False)